# SASRec Time-Aware BPI2012 Colab Train (`refine_ml50_do035` baseline)

Colab notebook for the next time-aware SASRec experiment using `refine_ml50_do035` as the fixed baseline.

Goals:
- reuse the completed `refine_ml50_do035` baseline results
- train only the new time-aware runs
- compare baseline vs `8-bucket` vs `9-bucket`
- evaluate under both `NDCG@10` and `NDCG@5` model-selection criteria


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
TIMEAWARE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10'
TIMEAWARE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('TIMEAWARE_NDCG10_OUTPUT_DIR:', TIMEAWARE_NDCG10_OUTPUT_DIR)
print('TIMEAWARE_NDCG5_OUTPUT_DIR:', TIMEAWARE_NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
BASELINE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5
TIMEAWARE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10
TIMEAWARE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG10_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG5_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


In [8]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Experiment design

Fixed baseline setting:
- `refine_ml50_do035`
- `hidden_units=50, num_blocks=2, num_heads=1, maxlen=50, lr=0.001, dropout=0.35`
- seeds: `42`, `2024`, `7`

Comparison targets:
- baseline (reuse existing completed runs)
- time-aware `8-bucket`
- time-aware `9-bucket`

Time-aware design:
- `x = item_embedding + positional_embedding + time_embedding`
- time source: `delta_prev_seconds`


## Check existing baseline runs

These baseline runs should already exist and must not be retrained.


In [10]:
from pathlib import Path

baseline_ndcg10_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
baseline_ndcg5_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

for label, output_dir, run_names in [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR), baseline_ndcg10_runs),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR), baseline_ndcg5_runs),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS
Baseline NDCG@5
refine_ml50_do035_s42 EXISTS
refine_ml50_do035_s2024 EXISTS
refine_ml50_do035_s7 EXISTS


## Check planned time-aware runs

Only train runs that are still missing.


In [11]:
planned_ndcg10 = [
    'timeaware_refine_ml50_do035_b8_s42',
    'timeaware_refine_ml50_do035_b8_s2024',
    'timeaware_refine_ml50_do035_b8_s7',
    'timeaware_refine_ml50_do035_b9_s42',
    'timeaware_refine_ml50_do035_b9_s2024',
    'timeaware_refine_ml50_do035_b9_s7',
]
planned_ndcg5 = [
    'timeaware_refine_ml50_do035_b8_s42',
    'timeaware_refine_ml50_do035_b8_s2024',
    'timeaware_refine_ml50_do035_b8_s7',
    'timeaware_refine_ml50_do035_b9_s42',
    'timeaware_refine_ml50_do035_b9_s2024',
    'timeaware_refine_ml50_do035_b9_s7',
]

for label, output_dir, run_names in [
    ('Time-aware NDCG@10', Path(TIMEAWARE_NDCG10_OUTPUT_DIR), planned_ndcg10),
    ('Time-aware NDCG@5', Path(TIMEAWARE_NDCG5_OUTPUT_DIR), planned_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Time-aware NDCG@10
timeaware_refine_ml50_do035_b8_s42 EXISTS
timeaware_refine_ml50_do035_b8_s2024 EXISTS
timeaware_refine_ml50_do035_b8_s7 OK
timeaware_refine_ml50_do035_b9_s42 OK
timeaware_refine_ml50_do035_b9_s2024 OK
timeaware_refine_ml50_do035_b9_s7 OK
Time-aware NDCG@5
timeaware_refine_ml50_do035_b8_s42 OK
timeaware_refine_ml50_do035_b8_s2024 OK
timeaware_refine_ml50_do035_b8_s7 OK
timeaware_refine_ml50_do035_b9_s42 OK
timeaware_refine_ml50_do035_b9_s2024 OK
timeaware_refine_ml50_do035_b9_s7 OK


## Train time-aware runs for `NDCG@10`

Run these cells only if the corresponding run directory does not already exist.


### timeaware_refine_ml50_do035_b8_s42


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b8_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10/timeaware_refine_ml50_do035_b8_s42
epoch=1, loss=0.5829
epoch=2, loss=0.2632
epoch=3, loss=0.1991
epoch=4, loss=0.1635
epoch=5, loss=0.1403
valid [full], NDCG@5: 0.6186, HR@5: 0.6949, NDCG@10: 0.6822, HR@10: 0.8911, MRR: 0.6297
valid [sampled], NDCG@5: 0.5559, HR@5: 0.5598, NDCG@10: 0.5643, HR@10: 0.5868, MRR: 0.5701
test [full], NDCG@5: 0.5853, HR@5: 0.7508, NDCG@10: 0.6262, HR@10: 0.8816, MRR: 0.5557
test [sampled], NDCG@5: 0.2091, HR@5: 0.2126, NDCG@10: 0.2428, HR@10: 0.3219, MRR: 0.2514
saved eval checkpoint: /content/drive/MyDrive/ai-project

### timeaware_refine_ml50_do035_b8_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b8_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10/timeaware_refine_ml50_do035_b8_s2024
epoch=1, loss=0.6099
epoch=2, loss=0.2708
epoch=3, loss=0.1978
epoch=4, loss=0.1583
epoch=5, loss=0.1377
valid [full], NDCG@5: 0.7021, HR@5: 0.8905, NDCG@10: 0.7366, HR@10: 0.9919, MRR: 0.6577
valid [sampled], NDCG@5: 0.5478, HR@5: 0.5585, NDCG@10: 0.5607, HR@10: 0.5989, MRR: 0.5657
test [full], NDCG@5: 0.7724, HR@5: 0.9148, NDCG@10: 0.8008, HR@10: 1.0000, MRR: 0.7379
test [sampled], NDCG@5: 0.2150, HR@5: 0.2771, NDCG@10: 0.2735, HR@10: 0.4577, MRR: 0.2478
saved eval checkpoint: /content/drive/MyDrive/ai-proje

### timeaware_refine_ml50_do035_b8_s7


In [12]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b8_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10/timeaware_refine_ml50_do035_b8_s7
epoch=1, loss=0.5810
epoch=2, loss=0.2715
epoch=3, loss=0.2022
epoch=4, loss=0.1681
epoch=5, loss=0.1452
valid [full], NDCG@5: 0.6743, HR@5: 0.8234, NDCG@10: 0.7235, HR@10: 0.9740, MRR: 0.6495
valid [sampled], NDCG@5: 0.5617, HR@5: 0.5622, NDCG@10: 0.5672, HR@10: 0.5800, MRR: 0.5797
test [full], NDCG@5: 0.6352, HR@5: 0.8429, NDCG@10: 0.6734, HR@10: 0.9623, MRR: 0.5850
test [sampled], NDCG@5: 0.1949, HR@5: 0.2005, NDCG@10: 0.2382, HR@10: 0.3405, MRR: 0.2418
saved eval checkpoint: /content/drive/MyDrive/ai-projects

### timeaware_refine_ml50_do035_b9_s42


In [13]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10/timeaware_refine_ml50_do035_b9_s42
epoch=1, loss=0.5811
epoch=2, loss=0.2632
epoch=3, loss=0.1985
epoch=4, loss=0.1625
epoch=5, loss=0.1395
valid [full], NDCG@5: 0.6291, HR@5: 0.7182, NDCG@10: 0.6887, HR@10: 0.9047, MRR: 0.6329
valid [sampled], NDCG@5: 0.5580, HR@5: 0.5632, NDCG@10: 0.5672, HR@10: 0.5920, MRR: 0.5722
test [full], NDCG@5: 0.5805, HR@5: 0.7521, NDCG@10: 0.6231, HR@10: 0.8896, MRR: 0.5485
test [sampled], NDCG@5: 0.2080, HR@5: 0.2099, NDCG@10: 0.2296, HR@10: 0.2808, MRR: 0.2495
saved eval checkpoint: /content/drive/MyDrive/ai-project

### timeaware_refine_ml50_do035_b9_s2024


In [14]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10/timeaware_refine_ml50_do035_b9_s2024
epoch=1, loss=0.6064
epoch=2, loss=0.2697
epoch=3, loss=0.1967
epoch=4, loss=0.1575
epoch=5, loss=0.1370
valid [full], NDCG@5: 0.7001, HR@5: 0.8924, NDCG@10: 0.7348, HR@10: 0.9924, MRR: 0.6552
valid [sampled], NDCG@5: 0.5457, HR@5: 0.5543, NDCG@10: 0.5581, HR@10: 0.5935, MRR: 0.5643
test [full], NDCG@5: 0.7559, HR@5: 0.8925, NDCG@10: 0.7916, HR@10: 1.0000, MRR: 0.7266
test [sampled], NDCG@5: 0.2105, HR@5: 0.2729, NDCG@10: 0.2679, HR@10: 0.4498, MRR: 0.2422
saved eval checkpoint: /content/drive/MyDrive/ai-proje

### timeaware_refine_ml50_do035_b9_s7


In [15]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg10/timeaware_refine_ml50_do035_b9_s7
epoch=1, loss=0.5822
epoch=2, loss=0.2708
epoch=3, loss=0.2024
epoch=4, loss=0.1677
epoch=5, loss=0.1449
valid [full], NDCG@5: 0.6688, HR@5: 0.8131, NDCG@10: 0.7220, HR@10: 0.9727, MRR: 0.6481
valid [sampled], NDCG@5: 0.5631, HR@5: 0.5635, NDCG@10: 0.5691, HR@10: 0.5833, MRR: 0.5807
test [full], NDCG@5: 0.6522, HR@5: 0.8841, NDCG@10: 0.6811, HR@10: 0.9731, MRR: 0.5887
test [sampled], NDCG@5: 0.1896, HR@5: 0.1951, NDCG@10: 0.2325, HR@10: 0.3341, MRR: 0.2387
saved eval checkpoint: /content/drive/MyDrive/ai-projects

## Train time-aware runs for `NDCG@5`

Run these cells only if the corresponding run directory does not already exist.


### timeaware_refine_ml50_do035_b8_s42


In [16]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b8_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5/timeaware_refine_ml50_do035_b8_s42
epoch=1, loss=0.5829
epoch=2, loss=0.2632
epoch=3, loss=0.1991
epoch=4, loss=0.1635
epoch=5, loss=0.1403
valid [full], NDCG@5: 0.6186, HR@5: 0.6949, NDCG@10: 0.6822, HR@10: 0.8911, MRR: 0.6297
valid [sampled], NDCG@5: 0.5559, HR@5: 0.5598, NDCG@10: 0.5643, HR@10: 0.5868, MRR: 0.5701
test [full], NDCG@5: 0.5853, HR@5: 0.7508, NDCG@10: 0.6262, HR@10: 0.8816, MRR: 0.5557
test [sampled], NDCG@5: 0.2091, HR@5: 0.2126, NDCG@10: 0.2428, HR@10: 0.3219, MRR: 0.2514
saved eval checkpoint: /content/drive/MyDrive/ai-projects/

### timeaware_refine_ml50_do035_b8_s2024


In [17]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b8_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5/timeaware_refine_ml50_do035_b8_s2024
epoch=1, loss=0.6099
epoch=2, loss=0.2708
epoch=3, loss=0.1978
epoch=4, loss=0.1583
epoch=5, loss=0.1377
valid [full], NDCG@5: 0.7021, HR@5: 0.8905, NDCG@10: 0.7366, HR@10: 0.9919, MRR: 0.6577
valid [sampled], NDCG@5: 0.5478, HR@5: 0.5585, NDCG@10: 0.5607, HR@10: 0.5989, MRR: 0.5657
test [full], NDCG@5: 0.7724, HR@5: 0.9148, NDCG@10: 0.8008, HR@10: 1.0000, MRR: 0.7379
test [sampled], NDCG@5: 0.2150, HR@5: 0.2771, NDCG@10: 0.2735, HR@10: 0.4577, MRR: 0.2478
saved eval checkpoint: /content/drive/MyDrive/ai-project

### timeaware_refine_ml50_do035_b8_s7


In [18]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b8_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5/timeaware_refine_ml50_do035_b8_s7
epoch=1, loss=0.5810
epoch=2, loss=0.2715
epoch=3, loss=0.2022
epoch=4, loss=0.1681
epoch=5, loss=0.1452
valid [full], NDCG@5: 0.6743, HR@5: 0.8234, NDCG@10: 0.7235, HR@10: 0.9740, MRR: 0.6495
valid [sampled], NDCG@5: 0.5617, HR@5: 0.5622, NDCG@10: 0.5672, HR@10: 0.5800, MRR: 0.5797
test [full], NDCG@5: 0.6352, HR@5: 0.8429, NDCG@10: 0.6734, HR@10: 0.9623, MRR: 0.5850
test [sampled], NDCG@5: 0.1949, HR@5: 0.2005, NDCG@10: 0.2382, HR@10: 0.3405, MRR: 0.2418
saved eval checkpoint: /content/drive/MyDrive/ai-projects/t

### timeaware_refine_ml50_do035_b9_s42


In [19]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b9_s42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5/timeaware_refine_ml50_do035_b9_s42
epoch=1, loss=0.5811
epoch=2, loss=0.2632
epoch=3, loss=0.1985
epoch=4, loss=0.1625
epoch=5, loss=0.1395
valid [full], NDCG@5: 0.6291, HR@5: 0.7182, NDCG@10: 0.6887, HR@10: 0.9047, MRR: 0.6329
valid [sampled], NDCG@5: 0.5580, HR@5: 0.5632, NDCG@10: 0.5672, HR@10: 0.5920, MRR: 0.5722
test [full], NDCG@5: 0.5805, HR@5: 0.7521, NDCG@10: 0.6231, HR@10: 0.8896, MRR: 0.5485
test [sampled], NDCG@5: 0.2080, HR@5: 0.2099, NDCG@10: 0.2296, HR@10: 0.2808, MRR: 0.2495
saved eval checkpoint: /content/drive/MyDrive/ai-projects/

### timeaware_refine_ml50_do035_b9_s2024


In [20]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b9_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5/timeaware_refine_ml50_do035_b9_s2024
epoch=1, loss=0.6064
epoch=2, loss=0.2697
epoch=3, loss=0.1967
epoch=4, loss=0.1575
epoch=5, loss=0.1370
valid [full], NDCG@5: 0.7001, HR@5: 0.8924, NDCG@10: 0.7348, HR@10: 0.9924, MRR: 0.6552
valid [sampled], NDCG@5: 0.5457, HR@5: 0.5543, NDCG@10: 0.5581, HR@10: 0.5935, MRR: 0.5643
test [full], NDCG@5: 0.7559, HR@5: 0.8925, NDCG@10: 0.7916, HR@10: 1.0000, MRR: 0.7266
test [sampled], NDCG@5: 0.2105, HR@5: 0.2729, NDCG@10: 0.2679, HR@10: 0.4498, MRR: 0.2422
saved eval checkpoint: /content/drive/MyDrive/ai-project

### timeaware_refine_ml50_do035_b9_s7


In [21]:
!python src/train_sasrec.py \
  --run_name timeaware_refine_ml50_do035_b9_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_refine_ml50_do035_ndcg5/timeaware_refine_ml50_do035_b9_s7
epoch=1, loss=0.5822
epoch=2, loss=0.2708
epoch=3, loss=0.2024
epoch=4, loss=0.1677
epoch=5, loss=0.1449
valid [full], NDCG@5: 0.6688, HR@5: 0.8131, NDCG@10: 0.7220, HR@10: 0.9727, MRR: 0.6481
valid [sampled], NDCG@5: 0.5631, HR@5: 0.5635, NDCG@10: 0.5691, HR@10: 0.5833, MRR: 0.5807
test [full], NDCG@5: 0.6522, HR@5: 0.8841, NDCG@10: 0.6811, HR@10: 0.9731, MRR: 0.5887
test [sampled], NDCG@5: 0.1896, HR@5: 0.1951, NDCG@10: 0.2325, HR@10: 0.3341, MRR: 0.2387
saved eval checkpoint: /content/drive/MyDrive/ai-projects/t

## Rebuild result tables from run folders

This avoids schema issues and lets us combine existing baseline runs with new time-aware runs safely.


In [22]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'time_bucket_boundaries': ','.join(str(x) for x in config.get('time_bucket_boundaries', [])),
            'time_bucket_count': config.get('time_bucket_count'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


In [23]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

## NDCG@10 comparison summary


In [24]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
timeaware_runs = [
    'timeaware_refine_ml50_do035_b8_s42',
    'timeaware_refine_ml50_do035_b8_s2024',
    'timeaware_refine_ml50_do035_b8_s7',
    'timeaware_refine_ml50_do035_b9_s42',
    'timeaware_refine_ml50_do035_b9_s2024',
    'timeaware_refine_ml50_do035_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['bucket_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['bucket_variant'] = timeaware_subset['run_name'].apply(lambda x: 'b8' if '_b8_' in x else 'b9')

df_ndcg10 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['bucket_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'bucket_variant', 'use_time_embedding', 'time_bucket_boundaries',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,bucket_variant,use_time_embedding,time_bucket_boundaries,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,timeaware_refine_ml50_do035_b8_s7,7,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.755675,0.961206,0.718767,0.845634,0.695393,0.820329,1.000000,0.820236,0.999729,0.760042,0.620062,0.686413,0.596447,0.612228,0.613063,0.355101,0.473449,0.307582,0.322270,0.355078
1,timeaware_refine_ml50_do035_b8_s42,42,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.735872,0.950223,0.699404,0.834573,0.673375,0.825379,1.000000,0.807890,0.943658,0.767326,0.581745,0.647243,0.557606,0.571643,0.576266,0.398008,0.547966,0.341492,0.369438,0.381958
2,timeaware_refine_ml50_do035_b8_s2024,2024,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.737589,0.975212,0.703871,0.869460,0.665396,0.802276,0.999865,0.793266,0.973869,0.737286,0.564312,0.626979,0.541552,0.555676,0.562416,0.300249,0.426269,0.254670,0.282194,0.297626
3,timeaware_refine_ml50_do035_b9_s7,7,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.754079,0.960665,0.710260,0.825358,0.693503,0.835758,0.999865,0.835715,0.999729,0.782300,0.617529,0.683424,0.593645,0.608152,0.610860,0.476189,0.573015,0.437701,0.451775,0.473501
4,timeaware_refine_ml50_do035_b9_s42,42,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.732318,0.970700,0.685321,0.827184,0.662175,0.752063,1.000000,0.738352,0.959405,0.667788,0.579360,0.618615,0.565768,0.576616,0.581879,0.270763,0.455421,0.197753,0.221875,0.251691
5,timeaware_refine_ml50_do035_b9_s2024,2024,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.738726,0.973596,0.705716,0.869190,0.667216,0.805208,0.999865,0.789474,0.949499,0.739477,0.566021,0.634556,0.540436,0.553646,0.562766,0.315404,0.474746,0.255381,0.283819,0.301400
6,refine_ml50_do035_s7,7,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.000000,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750
7,refine_ml50_do035_s42,42,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.000000,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
8,refine_ml50_do035_s2024,2024,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.000000,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183


In [25]:
summary_ndcg10 = df_ndcg10.groupby('bucket_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                  mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
bucket_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
b8                            0.743045  0.010971              0.962214  0.012525               0.707348  0.010139             0.849889  0.017828            0.678055  0.015536               0.815995  0.012146             0.999955  0.000078              0.807131  0.013501            0.972419  0.028064           0.754885  0.015670                   0.588706  0.028520                 0.653545  0.030214                  0.565201  0.028225                0.579849  0.029156               0.583915  0.026175                  0.351119  0.049001                0.482561  0.061358                 0.301248  0.043756               0.324634  0.043670              0.344887  0.043079
b9                            0.741707  0.011182              0.968320  0.006786               0.700432  0.013283             0.840577  0.024796            0.674298  0.016822               0.797676  0.042353             0.999910  0.000078              0.787847  0.048702            0.969544  0.026606           0.729855  0.057859                   0.587637  0.026733                 0.645532  0.033770                  0.566616  0.026614                0.579471  0.027365               0.585168  0.024215                  0.354119  0.108046                0.501061  0.063059                 0.296945  0.125257               0.319156  0.118954              0.342198  0.116397
baseline                      0.735259  0.006374              0.977329  0.015618               0.702263  0.009451             0.872599  0.029271            0.662771  0.002053               0.846684  0.062093             1.000000  0.000000              0.818405  0.078916            0.911416  0.055280           0.798923  0.080599                   0.572973  0.004280                 0.604843  0.007136                  0.561597  0.005491                0.569159  0.005422               0.579235  0.005635                  0.372338  0.102528                0.576445  0.125558                 0.309277  0.098875               0.383139  0.1057

Interpretation guide for NDCG@10:
- first compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` across `baseline`, `b8`, and `b9`
- then check whether sampled metrics and MRR show a similar trend
- baseline is reused from the completed sanity-check candidate; only time-aware runs are newly trained here


## NDCG@5 comparison summary


In [26]:
baseline_runs = [
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]
timeaware_runs = [
    'timeaware_refine_ml50_do035_b8_s42',
    'timeaware_refine_ml50_do035_b8_s2024',
    'timeaware_refine_ml50_do035_b8_s7',
    'timeaware_refine_ml50_do035_b9_s42',
    'timeaware_refine_ml50_do035_b9_s2024',
    'timeaware_refine_ml50_do035_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['bucket_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['bucket_variant'] = timeaware_subset['run_name'].apply(lambda x: 'b8' if '_b8_' in x else 'b9')

df_ndcg5 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['bucket_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'bucket_variant', 'use_time_embedding', 'time_bucket_boundaries',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,bucket_variant,use_time_embedding,time_bucket_boundaries,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,timeaware_refine_ml50_do035_b8_s7,7,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.755675,0.961206,0.718767,0.845634,0.695393,0.820329,1.000000,0.820236,0.999729,0.760042,0.620062,0.686413,0.596447,0.612228,0.613063,0.355101,0.473449,0.307582,0.322270,0.355078
1,timeaware_refine_ml50_do035_b8_s42,42,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.735872,0.950223,0.699404,0.834573,0.673375,0.825379,1.000000,0.807890,0.943658,0.767326,0.581745,0.647243,0.557606,0.571643,0.576266,0.398008,0.547966,0.341492,0.369438,0.381958
2,timeaware_refine_ml50_do035_b8_s2024,2024,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.732427,0.971374,0.707124,0.888889,0.660945,0.802967,1.000000,0.787603,0.952297,0.736190,0.566173,0.608890,0.551806,0.564033,0.569987,0.253582,0.474312,0.169535,0.207130,0.223758
3,timeaware_refine_ml50_do035_b9_s7,7,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.747653,0.961280,0.717019,0.864614,0.683025,0.821596,1.000000,0.821551,0.999864,0.761243,0.583207,0.673001,0.548985,0.564952,0.572282,0.416175,0.516190,0.377707,0.393849,0.418359
4,timeaware_refine_ml50_do035_b9_s42,42,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.732318,0.970700,0.685321,0.827184,0.662175,0.752063,1.000000,0.738352,0.959405,0.667788,0.579360,0.618615,0.565768,0.576616,0.581879,0.270763,0.455421,0.197753,0.221875,0.251691
5,timeaware_refine_ml50_do035_b9_s2024,2024,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.738726,0.973596,0.705716,0.869190,0.667216,0.805208,0.999865,0.789474,0.949499,0.739477,0.566021,0.634556,0.540436,0.553646,0.562766,0.315404,0.474746,0.255381,0.283819,0.301400
6,refine_ml50_do035_s7,7,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.728826,0.961737,0.696223,0.857259,0.660451,0.872416,1.000000,0.859037,0.957678,0.831208,0.577254,0.611043,0.565802,0.575315,0.581977,0.403018,0.656933,0.324827,0.417493,0.345750
7,refine_ml50_do035_s42,42,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.735378,0.977276,0.697410,0.854186,0.663510,0.891774,1.000000,0.868726,0.926375,0.858375,0.568693,0.606441,0.555385,0.565093,0.572753,0.456024,0.640631,0.399455,0.467411,0.419230
8,refine_ml50_do035_s2024,2024,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.741573,0.992973,0.713154,0.906351,0.664353,0.775861,1.000000,0.727453,0.850196,0.707187,0.572972,0.597043,0.563603,0.567069,0.582973,0.257972,0.431769,0.203548,0.264514,0.234183


In [27]:
summary_ndcg5 = df_ndcg5.groupby('bucket_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                  mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
bucket_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
b8                            0.741325  0.012546              0.960934  0.010578               0.708432  0.009748             0.856365  0.028704            0.676571  0.017445               0.816225  0.011757             1.000000  0.000000              0.805243  0.016477            0.965228  0.030189           0.754519  0.016286                   0.589327  0.027733                 0.647516  0.038762                  0.568620  0.024273                0.582635  0.025910               0.586438  0.023270                  0.335564  0.074169                0.498576  0.042775                 0.272870  0.091082               0.299613  0.083492              0.320265  0.084651
b9                            0.739566  0.007702              0.968525  0.006439               0.702685  0.016065             0.853663  0.023045            0.670805  0.010879               0.792956  0.036350             0.999955  0.000078              0.783126  0.041961            0.969589  0.026683           0.722836  0.048900                   0.576196  0.009019                 0.642057  0.027958                  0.551730  0.012887                0.565071  0.011485               0.572309  0.009557                  0.334114  0.074490                0.482119  0.031048                 0.276947  0.091895               0.299848  0.087100              0.323817  0.085565
baseline                      0.735259  0.006374              0.977329  0.015618               0.702263  0.009451             0.872599  0.029271            0.662771  0.002053               0.846684  0.062093             1.000000  0.000000              0.818405  0.078916            0.911416  0.055280           0.798923  0.080599                   0.572973  0.004280                 0.604843  0.007136                  0.561597  0.005491                0.569159  0.005422               0.579235  0.005635                  0.372338  0.102528                0.576445  0.125558                 0.309277  0.098875               0.383139  0.1057

Interpretation guide for NDCG@5:
- first compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` across `baseline`, `b8`, and `b9`
- then check whether sampled metrics and MRR show a similar trend
- baseline is reused from the completed sanity-check candidate; only time-aware runs are newly trained here
